<a href="https://colab.research.google.com/github/aliya-fatma011/Deep-Learning/blob/main/Cassava(ShuffleNet1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset_path = '/content/'

In [ ]:
!pip install -q tensorflow pandas scikit-learn matplotlib

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model

print("TensorFlow:", tf.__version__)

TensorFlow: 2.20.0


In [ ]:
import os

# Ensure the dataset is unzipped before listing its contents
!unzip -q "/content/drive/MyDrive/main_dataset.zip" -d "/content/"

print(os.listdir(dataset_path))

FileNotFoundError: [Errno 2] No such file or directory: '/content/main_dataset'

### Error: File Not Found

The `FileNotFoundError` indicates that the path specified for your dataset (`/content/drive/MyDrive/dataset_sir/main_dataset`) does not exist or is incorrect. This often happens if the folder names or structure in your Google Drive do not exactly match the path defined in the notebook.

Please verify the exact path to your `main_dataset` folder in your Google Drive. You can use the cell below to list the contents of your `/content/drive/MyDrive` folder to help you locate it.

Once you find the correct path, you will need to update the `dataset_path` variable in cell `ZWCTg7NqPRm9` and the `BASE_PATH` variable in cell `T1VXQRz-RR6i`.

In [ ]:
print("Contents of /content/drive/MyDrive:")
!ls -F "/content/drive/MyDrive/"

Contents of /content/drive/MyDrive:
'Colab Notebooks'/	   'Untitled form.gform'
 file.pdf		   'Untitled project.gscript'
 main_dataset.zip	   'Untitled spreadsheet (1).gsheet'
 Products.gsheet	   'Untitled spreadsheet (2).gsheet'
 Products.pdf		   'Untitled spreadsheet (3).gsheet'
'Untitled form (1).gform'  'Untitled spreadsheet.gsheet'


In [ ]:
# Unzip the main_dataset.zip file
!unzip -q "/content/drive/MyDrive/main_dataset.zip" -d "/content/"

replace /content/dataset_sir/main_dataset.zip? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
print("Contents of /content/:")
!ls -F "/content/"

In [ ]:
import pandas as pd

In [ ]:
BASE_PATH = '/content/'

TRAIN_CSV = BASE_PATH + "Train/Train.csv"
TRAIN_DIR = BASE_PATH + "Train/Train_Images"

TEST_CSV = BASE_PATH + "Test/Test.csv"
TEST_DIR = BASE_PATH + "Test/Test_images"

LABEL_JSON = BASE_PATH + "label_num_to_disease_map.json"

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

with open(LABEL_JSON, "r") as f:
    label_map = json.load(f)

print("Train data:")
print(train_df.head())

print("Test data:")
print(test_df.head())

print("Label map:")
print(label_map)

print("Train columns:", train_df.columns)
print("Test columns:", test_df.columns)


FileNotFoundError: [Errno 2] No such file or directory: '/content/main_dataset/Train/Train.csv'

#Counts the Total Numbers of Labels

In [ ]:
train_df["Labels"] = train_df["Labels"].astype(int)
test_df["Labels"] = test_df["Labels"].astype(int)

num_classes = len(label_map)

print("Number of classes:", num_classes)
print("Train class count:")
print(train_df["Labels"].value_counts())

#Counts Total numbers of Taining and Test Images

In [ ]:
print("Number of classes:", num_classes)

print("\nTotal images:")
print("Train:", len(train_df))
print("Test:", len(test_df))

print("\nClass distribution (Train):")
print(train_df["Labels"].value_counts())

#label mapping

In [ ]:
import json

with open(LABEL_JSON, 'r') as f:
    label_map = json.load(f)

print(label_map)

Convert labels → actual names

In [ ]:
for k, v in label_map.items():
    print(f"Label {k} → {v}")



In [ ]:
counts = train_df["Labels"].value_counts().sort_index()

for label, count in counts.items():
    print(f"{label_map[str(label)]} → {count}")

In [ ]:
print("Train Class Distribution (from DataFrame):")
counts_from_df = train_df["Labels"].value_counts().sort_index()
for label, count in counts_from_df.items():
    print(f"{label_map[str(label)]} → {count}")

In [ ]:
import os

train_dir = '/content/drive/MyDrive/dataset_sir/main_dataset/Train'
test_dir = '/content/drive/MyDrive/dataset_sir/main_dataset/Test'

def count_images(folder):
    total = 0
    for root, dirs, files in os.walk(folder):
        images = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        total += len(images)
    return total

print("Total Train Images:", count_images(train_dir))
print("Total Test Images:", count_images(test_dir))

#Plotting the Barplot to see all the class image

In [ ]:
counts = train_df["Labels"].value_counts().sort_index()
names = [label_map[str(i)] for i in counts.index]
values = counts.values

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.bar(names, values)

plt.title("Cassava Disease Class Distribution")
plt.xlabel("Disease Type")
plt.ylabel("Number of Images")

plt.xticks(rotation=45)

# Add values on top
for i, v in enumerate(values):
    plt.text(i, v + 5, str(v), ha='center')

plt.show()

#Function to Access the single images

In [ ]:
print(train_df["Labels"].unique())
print(train_df["Labels"].dtype)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

def show_one_image(class_id):
    row = train_df[train_df["Labels"] == class_id].iloc[0]

    img_path = os.path.join(TRAIN_DIR, row["ID"])
    img = Image.open(img_path)

    plt.imshow(img)
    plt.title(label_map[str(class_id)])
    plt.axis("off")
    plt.show()

In [ ]:
show_one_image(0)   # change class id

In [ ]:
show_one_image(1)   # change class id

In [ ]:
show_one_image(2)   # change class id

In [ ]:
show_one_image(3)   # change class id

In [ ]:
show_one_image(4)   # change class id

#Function to Access 3 random images from EACH class

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

def show_random_images_per_class(n=3):
    classes = sorted(train_df["Labels"].unique())

    plt.figure(figsize=(15, len(classes) * 3))

    for i, class_id in enumerate(classes):
        rows = train_df[train_df["Labels"] == class_id].sample(n)

        for j, (_, row) in enumerate(rows.iterrows()):
            img_path = os.path.join(TRAIN_DIR, row["ID"])
            img = Image.open(img_path)

            plt.subplot(len(classes), n, i*n + j + 1)
            plt.imshow(img)
            plt.title(label_map[str(class_id)])
            plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_random_images_per_class(n=3)

#Image preprocessing

In [ ]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

#Applying image preprocessing on CMD Disease of first image

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

# get one row
row = train_df.iloc[3]

# image path
image_path = os.path.join(TRAIN_DIR, row["ID"])

# label
label_id = row["Labels"]
label_name = label_map[str(label_id)]

# load image
img = Image.open(image_path)

# show image with ONLY class name
plt.imshow(img)
plt.title(label_name)
plt.axis("off")
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(img)
plt.title(f"Original Image - {label_name}")
plt.axis("off")
plt.show()

Resize Image

In [ ]:
def show_image(title, image, cmap=None):
    plt.figure(figsize=(5,5))
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
resized = cv2.resize(img, (224, 224))
show_image(f"Resized Image - {label_name}", resized)

Grayscale

In [ ]:
gray = cv2.cvtColor(resized, cv2.COLOR_RGB2GRAY)
show_image("Grayscale", gray, cmap="gray")



Gaussian BLur

In [ ]:
gaussian_blur = cv2.GaussianBlur(resized, (5, 5), 0)
show_image("Gaussian Blur", gaussian_blur)

Median Blur

In [ ]:
median_blur = cv2.medianBlur(resized, 5)
show_image("Median Blur", median_blur)

Sharpening

In [ ]:
sharpen_kernel = np.array([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
])

sharpened = cv2.filter2D(resized, -1, sharpen_kernel)
show_image("Sharpened Image", sharpened)

Histogram Equilization

In [ ]:
gray_equalized = cv2.equalizeHist(gray)
show_image("Histogram Equalized Grayscale", gray_equalized, cmap="gray")

CLAHE enhancement

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
clahe_img = clahe.apply(gray)

show_image("CLAHE Enhanced", clahe_img, cmap="gray")

Canny edge detection

In [ ]:
edges = cv2.Canny(gray, 100, 200)
show_image("Canny Edge Detection", edges, cmap="gray")

Sobel edge detection

In [ ]:
sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

sobel = cv2.magnitude(sobel_x, sobel_y)
sobel = np.uint8(np.clip(sobel, 0, 255))

show_image("Sobel Edge Detection", sobel, cmap="gray")




Thresholding

In [ ]:
_, threshold = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY)
show_image("Binary Threshold", threshold, cmap="gray")

In [ ]:
hsv = cv2.cvtColor(resized, cv2.COLOR_RGB2HSV)
h, s, v = cv2.split(hsv)

show_image("Hue", h, "gray")
show_image("Saturation", s, "gray")
show_image("Value", v, "gray")

In [ ]:
_, mask = cv2.threshold(s, 100, 255, cv2.THRESH_BINARY_INV)
show_image("Disease Mask", mask, "gray")

In [ ]:
highlight = resized.copy()
highlight[mask == 255] = [255, 0, 0]

show_image("Highlighted Disease Area", highlight)

In [ ]:
adaptive_threshold = cv2.adaptiveThreshold(
    gray,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    11,
    2
)

show_image("Adaptive Threshold", adaptive_threshold, cmap="gray")

Morphological operations

In [ ]:
kernel = np.ones((5,5), np.uint8)

erosion = cv2.erode(threshold, kernel, iterations=1)
dilation = cv2.dilate(threshold, kernel, iterations=1)
opening = cv2.morphologyEx(threshold, cv2.MORPH_OPEN, kernel)
closing = cv2.morphologyEx(threshold, cv2.MORPH_CLOSE, kernel)

show_image("Erosion", erosion, cmap="gray")
show_image("Dilation", dilation, cmap="gray")
show_image("Opening", opening, cmap="gray")
show_image("Closing", closing, cmap="gray")

HSV color space

In [ ]:
hsv = cv2.cvtColor(resized, cv2.COLOR_RGB2HSV)

h, s, v = cv2.split(hsv)

show_image("Hue Channel", h, cmap="gray")
show_image("Saturation Channel", s, cmap="gray")
show_image("Value Channel", v, cmap="gray")

RGB histogram

In [ ]:
colors = ("red", "green", "blue")

plt.figure(figsize=(8,5))

for i, color in enumerate(colors):
    hist = cv2.calcHist([resized], [i], None, [256], [0,256])
    plt.plot(hist, color=color)

plt.title("RGB Histogram")
plt.xlabel("Pixel Intensity")
plt.ylabel("Frequency")
plt.show()

Show everything in one grid

In [ ]:
images = [
    ("Original", resized, None),
    ("Grayscale", gray, "gray"),
    ("Gaussian Blur", gaussian_blur, None),
    ("Sharpened", sharpened, None),
    ("CLAHE", clahe_img, "gray"),
    ("Canny Edges", edges, "gray"),
    ("Sobel Edges", sobel, "gray"),
    ("Threshold", threshold, "gray"),
    ("Adaptive Threshold", adaptive_threshold, "gray")
]

plt.figure(figsize=(15, 12))

for i, (title, image, cmap) in enumerate(images):
    plt.subplot(3, 3, i + 1)
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

#Image Generator

In [ ]:
# Removed !pip install keras-shufflenet-v2 as it seems unavailable or problematic

In [ ]:
train_df = pd.read_csv('/content/dataset_sir/main_dataset/Train/Train.csv')
test_df  = pd.read_csv('/content/dataset_sir/main_dataset/Test/Test.csv')

# IMPORTANT → keep labels as strings for ImageDataGenerator sparse mode
train_df['Labels'] = train_df['Labels'].astype(str)
test_df['Labels']  = test_df['Labels'].astype(str)

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    rescale=1./255
)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='/content/dataset_sir/main_dataset/Train/Train_Images',
    x_col='ID',
    y_col='Labels',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    shuffle=True
)

val_gen = val_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory='/content/dataset_sir/main_dataset/Test/Test_images',
    x_col='ID',
    y_col='Labels',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    shuffle=False
)

In [ ]:
print(train_gen.class_indices)